This notebook lets the user find the air exchanges per hour neccecary to limit the probability of infection (as calculated by CAiMIRA) below a predefined threashold.

In [ ]:
import caimira.ventilation.scenarios as scenarios
import caimira.ventilation.find_air_exch as find_air_exch
import caimira.ventilation.plot_results as plot_results

In [ ]:
scenario = scenarios.ICU() # Scenarios tested: shared_office, classroom, patient_ward, ICU. No short range interactions.

lim_probability_infection_list = [0.1, 0.075, 0.05]
lim_probability_infection = lim_probability_infection_list[1]

In [ ]:
plot_results.plot_probabilities(scenario, lim_probability_infection_list)

In [ ]:
air_exch, probability = find_air_exch.find_constant_air_exch(scenario, lim_probability_infection)

NOTE:

Variability between model runs may cause the probability of infection to be slightly higher than lim_probability_infection.

In [ ]:
plot_results.plot_model_concentration_results(scenario, air_exch, deterministic_CO2=True, plot_clean_air_delivery=True)

In [ ]:
air_exch_l, vent_transition_t = find_air_exch.find_next_air_exch_by_co2(
    scenario,
    air_exch_list=[0.25],                                     # No ventilation
    vent_transition_times=None,                               # 
    max_CO2=800,                                              # Define a reasonable limit from the above plot (TODO: scale limit by number of occupants to make sense of scenarios where people come and go)
    min_CO2_fraction=0.9,                                     # Determines the lower limit for the CO2 concentration. Ventilation is decreased when reaching this point
    target_CO2_fraction=0.95,
    max_ventilation_changes=5
)

In [ ]:
for a, v in zip(air_exch_l, vent_transition_t):
    print(v, a)

In [ ]:
plot_results.plot_model_concentration_results(scenario, air_exch_l, vent_transition_t, deterministic_CO2=True)

NOTE: 

FEEDBACK CO2 THRESHOLD (propose a starting value from the profile, calculate P(I) and check if result is within limit, else set a lower value and reiterate)

When the number of ventilation changes is defined as less than the model's number of state changes, the CO2 concentration may increase beyond the max CO2 limit (and similarly below the min CO2 limit). 

This also means that the probability of infection is not neccecarily <= lim_probability_infection. 